## EDA Summary

- 79,902 rows, one per unique provider (`Rndrng_NPI`), no duplicates.
- Core financial columns (`Tot_Sbmtd_Chrg`, `Tot_Mdcr_Alowd_Amt`, `Tot_Mdcr_Pymt_Amt`) are complete and correctly typed as floats.
- Heavy missingness in demographic breakdown columns (`Bene_Race_*`, `Drug_Sprsn_Ind`, etc.) reflects CMS's privacy suppression policy for small patient counts (<11), not data quality issues.
- Average provider: ~$402K submitted vs. ~$116K Medicare-paid — a ~71% average markdown, forming the core financial narrative for this project.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/medicare_provider_ca.csv")
print("Shape:", df.shape)
df.columns.tolist()

Shape: (79902, 56)


['Rndrng_NPI',
 'Rndrng_Prvdr_Last_Org_Name',
 'Rndrng_Prvdr_First_Name',
 'Rndrng_Prvdr_MI',
 'Rndrng_Prvdr_Crdntls',
 'Rndrng_Prvdr_Ent_Cd',
 'Rndrng_Prvdr_St1',
 'Rndrng_Prvdr_St2',
 'Rndrng_Prvdr_City',
 'Rndrng_Prvdr_State_Abrvtn',
 'Rndrng_Prvdr_State_FIPS',
 'Rndrng_Prvdr_Zip5',
 'Rndrng_Prvdr_RUCA',
 'Rndrng_Prvdr_RUCA_Desc',
 'Rndrng_Prvdr_Cntry',
 'Rndrng_Prvdr_Type',
 'Rndrng_Prvdr_Mdcr_Prtcptg_Ind',
 'Tot_HCPCS_Cds',
 'Tot_Benes',
 'Tot_Srvcs',
 'Tot_Sbmtd_Chrg',
 'Tot_Mdcr_Alowd_Amt',
 'Tot_Mdcr_Pymt_Amt',
 'Tot_Mdcr_Stdzd_Amt',
 'Drug_Sprsn_Ind',
 'Drug_Tot_HCPCS_Cds',
 'Drug_Tot_Benes',
 'Drug_Tot_Srvcs',
 'Drug_Sbmtd_Chrg',
 'Drug_Mdcr_Alowd_Amt',
 'Drug_Mdcr_Pymt_Amt',
 'Drug_Mdcr_Stdzd_Amt',
 'Med_Sprsn_Ind',
 'Med_Tot_HCPCS_Cds',
 'Med_Tot_Benes',
 'Med_Tot_Srvcs',
 'Med_Sbmtd_Chrg',
 'Med_Mdcr_Alowd_Amt',
 'Med_Mdcr_Pymt_Amt',
 'Med_Mdcr_Stdzd_Amt',
 'Bene_Avg_Age',
 'Bene_Age_LT_65_Cnt',
 'Bene_Age_65_74_Cnt',
 'Bene_Age_75_84_Cnt',
 'Bene_Age_GT_84_Cnt',
 'Bene_Fe

In [2]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
Drug_Sprsn_Ind,71651,89.67
Med_Sprsn_Ind,71651,89.67
Bene_Race_Othr_Cnt,67034,83.90
Bene_Race_Black_Cnt,55429,69.37
Bene_Race_NatInd_Cnt,54517,68.23
Bene_Race_API_Cnt,51552,64.52
Rndrng_Prvdr_St2,42609,53.33
Bene_Race_Hspnc_Cnt,38804,48.56
Bene_Age_GT_84_Cnt,34379,43.03
Bene_Age_LT_65_Cnt,30552,38.24


In [3]:
df.dtypes

Rndrng_NPI                         int64
Rndrng_Prvdr_Last_Org_Name           str
Rndrng_Prvdr_First_Name              str
Rndrng_Prvdr_MI                      str
Rndrng_Prvdr_Crdntls                 str
Rndrng_Prvdr_Ent_Cd                  str
Rndrng_Prvdr_St1                     str
Rndrng_Prvdr_St2                     str
Rndrng_Prvdr_City                    str
Rndrng_Prvdr_State_Abrvtn            str
Rndrng_Prvdr_State_FIPS            int64
Rndrng_Prvdr_Zip5                  int64
Rndrng_Prvdr_RUCA                float64
Rndrng_Prvdr_RUCA_Desc               str
Rndrng_Prvdr_Cntry                   str
Rndrng_Prvdr_Type                    str
Rndrng_Prvdr_Mdcr_Prtcptg_Ind        str
Tot_HCPCS_Cds                      int64
Tot_Benes                          int64
Tot_Srvcs                        float64
Tot_Sbmtd_Chrg                   float64
Tot_Mdcr_Alowd_Amt               float64
Tot_Mdcr_Pymt_Amt                float64
Tot_Mdcr_Stdzd_Amt               float64
Drug_Sprsn_Ind  

In [4]:
print("Total rows:", len(df))
print("Unique NPIs:", df["Rndrng_NPI"].nunique())
print("Duplicate NPIs:", len(df) - df["Rndrng_NPI"].nunique())

Total rows: 79902
Unique NPIs: 79902
Duplicate NPIs: 0


In [5]:
df[["Tot_Sbmtd_Chrg", "Tot_Mdcr_Alowd_Amt", "Tot_Mdcr_Pymt_Amt"]].describe()

,Tot_Sbmtd_Chrg,Tot_Mdcr_Alowd_Amt,Tot_Mdcr_Pymt_Amt
count,7.990200e+04,7.990200e+04,7.990200e+04
mean,4.026222e+05,1.467871e+05,1.162731e+05
std,2.926017e+06,9.262315e+05,8.634285e+05
min,1.250000e+02,4.116000e+01,0.000000e+00
25%,2.803775e+04,1.331219e+04,1.018170e+04
50%,1.141745e+05,4.715637e+04,3.608323e+04
75%,3.312030e+05,1.294096e+05,9.929487e+04
max,3.686479e+08,1.660136e+08,1.660121e+08
